# 06 - Improved landmark and MobileNet model

In [ ]:
import sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.config import load_config
from src.improved_signer_cv import run_landmark_ensemble_cv
from src.robust_landmark_ensemble import run_experiment

config = load_config()
ablation_path = PROJECT_ROOT / 'outputs' / 'feature_ablation_cv' / 'feature_ablation_summary.csv'
if not ablation_path.exists():
    raise FileNotFoundError('Run notebook 07 first to select landmark features on training signers.')
feature_variant = pd.read_csv(ablation_path).iloc[0]['variant']
cv_summary = run_landmark_ensemble_cv(config, feature_variant=feature_variant)
selected_members = int(cv_summary.iloc[0]['model'].split('_')[1])
selection = {
    'indices': list(range(1, selected_members + 1)),
    'image_weight': 0.1,
    'fallback': True,
    'feature_variant': feature_variant,
}
print('Training-signer CV selection:', cv_summary.iloc[0].to_dict())
outcome = run_experiment(config, selection_override=selection)
print('Cross-validation-selected blend:', outcome['selection'])
print(f"Training time: {outcome['training_seconds']:.1f}s; total time: {outcome['total_seconds']:.1f}s")
print('Saved checkpoint:', outcome['checkpoint'])
